# Lesson 4: AI Paradigms

**Module 1:** Introduction to Artificial Intelligence

**Lesson Objectives:**
- Compare Expert Systems, Rule-Based, ML, Deep Learning, and LLMs
- Implement a rule-based system and a basic ML model
- Choose the appropriate paradigm for a given problem

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

print("Libraries loaded successfully")

## Paradigm 1: Rule-Based System

Let us build a medical symptom checker using explicit rules.

In [ ]:
def symptom_checker(symptoms):
    """Rule-based medical symptom checker."""
    symptom_set = set(s.lower() for s in symptoms)
    
    # Define rules
    if {"fever", "cough", "shortness_of_breath"}.issubset(symptom_set):
        return "Possible pneumonia — seek medical attention"
    if {"fever", "headache", "stiff_neck"}.issubset(symptom_set):
        return "Possible meningitis — URGENT"
    if {"fever", "rash", "joint_pain"}.issubset(symptom_set):
        return "Possible dengue fever"
    if {"fatigue", "weight_loss", "night_sweats"}.issubset(symptom_set):
        return "Possible tuberculosis"
    if {"sneezing", "runny_nose", "watery_eyes"}.issubset(symptom_set):
        return "Possible allergies"
    if {"fever", "cough", "fatigue"}.issubset(symptom_set):
        return "Possible common cold or flu"
    
    return "No matching condition found — consult a doctor"


# Test the rule-based system
test_cases = [
    ["fever", "cough", "shortness_of_breath"],
    ["fatigue", "weight_loss", "night_sweats"],
    ["fever", "headache"],  # Not enough symptoms
    ["sneezing", "runny_nose", "watery_eyes"],
]

print("Rule-Based Symptom Checker Results:")
print("-" * 50)
for case in test_cases:
    result = symptom_checker(case)
    print(f"Symptoms: {case}")
    print(f"  → {result}\n")

### Limitation: Brittleness

Notice that "fever + headache" returns no match. If we add "fever" and "cough" without "fatigue," the common cold rule does not fire. This is the **brittleness** of rule-based systems — they only match exact patterns.

## Paradigm 2: Machine Learning (Decision Tree)

Now let us train a Machine Learning model that learns rules from data instead of requiring hand-crafted rules.

In [ ]:
# Load the iris dataset
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Feature names: {feature_names}")
print(f"Target names: {target_names}")
print(f"\nFirst 5 rows:")
print(pd.DataFrame(X, columns=feature_names).head())

In [ ]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

In [ ]:
# Train a Decision Tree classifier
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)

# Predict on test set
y_pred = dt_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Decision Tree Accuracy: {accuracy:.2%}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(pd.DataFrame(cm, index=target_names, columns=target_names))

In [ ]:
# Visualize the decision tree
plt.figure(figsize=(15, 8))
plot_tree(dt_model, feature_names=feature_names, 
          class_names=target_names.tolist(), filled=True)
plt.title("Decision Tree for Iris Classification")
plt.show()

### Interpreting the Decision Tree

The Decision Tree learned rules automatically from the data:
- **First split**: petal length (<= 2.45 cm) → setosa vs. versicolor/virginica
- **Second split**: petal width and length further separate versicolor from virginica

These rules are similar to what an expert might write, but they were *learned from data* rather than hand-crafted.

## Paradigm 3: Simple Neural Network Concept

Let us demonstrate a single neuron (perceptron) — the building block of deep learning.

In [ ]:
class Neuron:
    """A single artificial neuron."""
    
    def __init__(self, n_inputs):
        self.weights = np.random.randn(n_inputs) * 0.1
        self.bias = np.random.randn() * 0.1
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, inputs):
        """Forward pass: compute output."""
        z = np.dot(inputs, self.weights) + self.bias
        return self.sigmoid(z)
    
    def __repr__(self):
        return f"Neuron(weights={self.weights.round(3)}, bias={self.bias:.3f})"


# Create a neuron and test it
n = Neuron(4)  # 4 inputs (like iris features)
sample = X_test[0]
output = n.forward(sample)

print(f"Input: {sample}")
print(f"Output: {output:.4f}")
print(f"Interpretation: Probability of class 1 = {output:.2%}")

## Hybrid System: Rules + ML

In production, systems often combine multiple paradigms. Here is a hybrid approach.

In [ ]:
def hybrid_predict(sepal_length, sepal_width, petal_length, petal_width):
    """
    Hybrid iris classifier.
    Rule-based for clear cases, ML for uncertain ones.
    """
    # Rule: very small petals = setosa
    if petal_length < 2.0:
        return ("setosa", "rule-based")
    
    # Rule: very large petals = virginica
    if petal_length > 5.5:
        return ("virginica", "rule-based")
    
    # Otherwise, use the ML model
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    pred = dt_model.predict(features)[0]
    return (target_names[pred], "ml")


# Test the hybrid system
test_samples = [
    (5.1, 3.5, 1.4, 0.2),  # Small petals → setosa (rule)
    (6.5, 3.0, 5.2, 2.0),  # Large petals → virginica (rule)
    (6.0, 2.7, 4.5, 1.5),  # Medium → ML
]

print("Hybrid Classifier Results:")
print("-" * 40)
for sample in test_samples:
    result, method = hybrid_predict(*sample)
    print(f"Input: {sample} → {result} (via {method})")

## Biotechnology Example: Protein Function Prediction

Compare paradigms for predicting protein function from sequence.

In [ ]:
# Simulated protein sequences and functions
proteins = [
    {"sequence": "MVLSPADKTNVKAAW", "motif": "heme_binding", "function": "Hemoglobin"},
    {"sequence": "MGHFTEEDKATITSL", "motif": "atp_binding", "function": "Kinase"},
    {"sequence": "MAKEGVKAVKAVLV", "motif": "dna_binding", "function": "Transcription factor"},
]

def rule_based_function(sequence):
    """Predict protein function using motifs."""
    if "heme" in sequence.lower():
        return "Hemoglobin (heme-binding protein)"
    if "atp" in sequence.lower():
        return "Kinase (ATP-binding protein)"
    if "dna" in sequence.lower():
        return "Transcription factor (DNA-binding protein)"
    return "Unknown function"


print("Rule-Based Protein Function Prediction:")
for p in proteins:
    pred = rule_based_function(p["sequence"])
    print(f"  {p['sequence']} → {pred}")

## SaaS Example: Customer Churn

Compare rule-based and ML approaches for churn prediction.

In [ ]:
# Generate synthetic customer data
np.random.seed(42)
n_customers = 200

customers = pd.DataFrame({
    'login_frequency': np.random.randint(0, 30, n_customers),
    'support_tickets': np.random.randint(0, 10, n_customers),
    'days_since_last_purchase': np.random.randint(0, 365, n_customers),
    'satisfaction_score': np.random.uniform(1, 5, n_customers),
})

def rule_based_churn(customer):
    """Rule-based churn prediction."""
    score = 0
    if customer['login_frequency'] < 5: score += 1
    if customer['support_tickets'] > 5: score += 1
    if customer['days_since_last_purchase'] > 180: score += 1
    if customer['satisfaction_score'] < 2.5: score += 1
    return score >= 2


# Apply rules
customers['churn_risk_rule'] = customers.apply(rule_based_churn, axis=1)

print("Rule-Based Churn Prediction (First 10 customers):")
print(customers.head(10))
print(f"\nCustomers at risk (rule): {customers['churn_risk_rule'].sum()} / {n_customers}")

## Exercises

1. Add 3 more rules to the symptom checker and test them.
2. Train a Random Forest classifier on the wine dataset and compare its accuracy to the Decision Tree.
3. Design a hybrid system for a problem of your choice — describe the rules and ML components.

In [ ]:
# Exercise 2: Random Forest on wine dataset
from sklearn.ensemble import RandomForestClassifier

wine = load_wine()
X_w, y_w = wine.data, wine.target

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    X_w, y_w, test_size=0.3, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(Xw_train, yw_train)
rf_pred = rf.predict(Xw_test)
rf_accuracy = accuracy_score(yw_test, rf_pred)

print(f"Random Forest Accuracy on Wine: {rf_accuracy:.2%}")

## Summary

- Five AI paradigms: Expert Systems, Rule-Based, ML, Deep Learning, LLMs
- Rule-Based: interpretable, no data, but brittle
- ML: learns from data, generalizes, needs examples
- Deep Learning: handles complex patterns, needs scale
- LLMs: language understanding, emergent abilities
- Hybrid systems combine the best of multiple paradigms